# **calt Minimal Demo**

This notebook demonstrates the minimal code needed to:
1. Generate a dataset
2. Train a model
3. Show evaluation results

In [ ]:
# import sys
# from pathlib import Path

# # Add development calt to path (prioritize over pip-installed version)
# # This notebook is in calt/examples/demos/, so we go up to calt/ and then to src/
# # When running in Jupyter, Path.cwd() gives the notebook's directory
# calt_dev_path = Path.cwd().parent.parent / "src"
# sys.path.insert(0, str(calt_dev_path))

# print(f"Using development calt from: {calt_dev_path}")

In [ ]:
%%capture
%pip install calt-x

In [ ]:
# On Colab: clone the repo so configs/ and data paths exist (skip if already present)
import os
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB and not os.path.exists("configs/data.yaml"):
    !git clone --depth 1 https://github.com/HiroshiKERA/calt.git
    os.chdir("calt/examples/demos")

print("Working directory:", os.getcwd())

## 1. Dataset Generation

<!-- Generate polynomial addition problems -->

In [ ]:
import random

from omegaconf import OmegaConf

from calt.dataset import DatasetPipeline
from calt.dataset.sympy.utils.polynomial_sampler import PolynomialSampler


# Define instance generator: polynomial addition
def polynomial_addition_generator(seed):
    random.seed(seed)

    # Initialize polynomial sampler
    sampler = PolynomialSampler(
        symbols="x0, x1",
        field_str="GF(7)",
        max_num_terms=2,
        max_degree=2,
        min_degree=1,
    )

    # Generate two polynomials
    F = sampler.sample(num_samples=2)

    # Solution is the sum
    g = sum(F)

    return F, g


# Load config from YAML file
cfg = OmegaConf.load("configs/data.yaml")

# Create dataset pipeline
pipeline = DatasetPipeline.from_config(
    cfg.dataset,
    instance_generator=polynomial_addition_generator,
)

# Run dataset generation
pipeline.run()

## 2. Model Training

<!-- Load data, create model, and train -->
<!-- The entire training pipeline can be summarized in just a few lines: -->


In [ ]:
# Complete minimal training code
from omegaconf import OmegaConf

from calt.io import IOPipeline
from calt.models import ModelPipeline
from calt.trainer import TrainerPipeline

# Load config from YAML file
cfg = OmegaConf.load("./configs/train.yaml")

# Load data
io_pipeline = IOPipeline.from_config(cfg.data)
result = io_pipeline.build()

# Create model
model = ModelPipeline(cfg.model, result["tokenizer"]).build()

# Create trainer and train
trainer = TrainerPipeline(
    cfg.train,
    model=model,
    tokenizer=result["tokenizer"],
    train_dataset=result["train_dataset"],
    eval_dataset=result["test_dataset"],
    data_collator=result["data_collator"],
).build()

trainer.train()
success_rate = trainer.evaluate_and_save_generation()
print(f"Success rate: {100 * success_rate:.1f}%")

## 3. Showing Evaluation Results

In [ ]:
from utils import showcase

# Show 10 success cases (requires running trainer.evaluate_and_save_generation())
showcase(result["test_dataset"], success_cases=True, num_show=10)

# Show 10 failure cases
showcase(result["test_dataset"], success_cases=False, num_show=10)